---
# **NLP Preprocessing Engine**
---

## **Task 1: Conceptual Understanding**
### *1. What is the difference between "Love" and "love" in NLP?*

- In NLP, "Love" and "love" are treated as different tokens if case-sensitive.
- This increases vocabulary size unnecessarily.
- Converting to lowercase ensures consistency.

### *2. What happens if stopwords are not removed?*
- Increases noise in data.
- Slows down processing.
- Reduces model accuracy.

### *3. Provide two real-world scenarios where removing stopwords can be harmful.*
- Sentiment Analysis
  - "I am NOT happy" -> removing "not" changes meaning
- Question Answering
  - "What is your name?" -> removing "what" breaks intent

### *4. Explain the difference between stemming and lemmatization.*
- Stemming: Cuts words roughly (running -> run)
- Lemmatization: Uses grammar rules (better accuracy)

---
## **Task 2: Build Advanced Preprocessing Function**

In [105]:
import re
from collections import Counter

def preprocess_text(text):
    try:
        # Handle invalid input
        if not text or not isinstance(text, str):
            return [], ""

        # 1. Lowercase
        text = text.lower()

        # 2. Remove URLs & emails
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)

        # 3. Remove emojis (non-ASCII)
        text = re.sub(r'[^\x00-\x7F]+', '', text)

        # 4. Remove numbers
        text = re.sub(r'\d+', '', text)

        # 5. Handle repeated characters (soooo → soo)
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)

        # 6. Remove special characters / punctuation
        text = re.sub(r'[^a-z\s]', '', text)

        # 7. Remove extra spaces
        text = re.sub(r'\s+', ' ', text).strip()

        # 8. Tokenization
        tokens = text.split()

        # 9. Remove short words (<=2), keep important ones
        tokens = [
            word for word in tokens
            if len(word) > 2 or word in ['no', 'not']
        ]

        # Final cleaned sentence
        cleaned_sentence = " ".join(tokens)

        return tokens, cleaned_sentence

    except Exception as e:
        print("Error:", e)
        return [], ""

---
## **Task 3: Stress Testing**

In [106]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://abc.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

for sentence in test_sentences:
    tokens, cleaned = preprocess_text(sentence)

    print("-" * 50)
    print(f"Original : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"Cleaned  : {cleaned}")

--------------------------------------------------
Original : Get 100% FREE access now!!!
Tokens   : ['get', 'free', 'access', 'now']
Cleaned  : get free access now
--------------------------------------------------
Original : I absolutely looooved this product 😍😍
Tokens   : ['absolutely', 'looved', 'this', 'product']
Cleaned  : absolutely looved this product
--------------------------------------------------
Original : Worst service ever... 0/10
Tokens   : ['worst', 'service', 'ever']
Cleaned  : worst service ever
--------------------------------------------------
Original : Call me at 9876543210
Tokens   : ['call']
Cleaned  : call
--------------------------------------------------
Original : This is THE best course!!!
Tokens   : ['this', 'the', 'best', 'course']
Cleaned  : this the best course
--------------------------------------------------
Original : Visit https://abc.com now!
Tokens   : ['visit', 'now']
Cleaned  : visit now
--------------------------------------------------
Orig

---
## **Task 4: Token Analytics**

In [107]:
def token_analysis(tokens):
    if not tokens:
        return 0, 0, 0

    total = len(tokens)
    unique = len(set(tokens))
    avg_len = sum(len(word) for word in tokens) / total

    return total, unique, avg_len


for sentence in test_sentences:
    tokens, _ = preprocess_text(sentence)
    total, unique, avg = token_analysis(tokens)

    print("-" * 50)
    print(f"Sentence : {sentence}")
    print(f"Tokens   : {tokens}")
    print(f"Total: {total} | Unique: {unique} | Avg Length: {round(avg,2)}")

--------------------------------------------------
Sentence : Get 100% FREE access now!!!
Tokens   : ['get', 'free', 'access', 'now']
Total: 4 | Unique: 4 | Avg Length: 4.0
--------------------------------------------------
Sentence : I absolutely looooved this product 😍😍
Tokens   : ['absolutely', 'looved', 'this', 'product']
Total: 4 | Unique: 4 | Avg Length: 6.75
--------------------------------------------------
Sentence : Worst service ever... 0/10
Tokens   : ['worst', 'service', 'ever']
Total: 3 | Unique: 3 | Avg Length: 5.33
--------------------------------------------------
Sentence : Call me at 9876543210
Tokens   : ['call']
Total: 1 | Unique: 1 | Avg Length: 4.0
--------------------------------------------------
Sentence : This is THE best course!!!
Tokens   : ['this', 'the', 'best', 'course']
Total: 4 | Unique: 4 | Avg Length: 4.25
--------------------------------------------------
Sentence : Visit https://abc.com now!
Tokens   : ['visit', 'now']
Total: 2 | Unique: 2 | Avg Le

## **Analysis Questions**

### *1. Which sentence had the most noise?*

- "Win $$$ now!!! Limited offer!!!"

- Reason:
This sentence contains excessive special characters, symbols, and promotional noise, making it highly unstructured.


### *2. Which sentence retained the most meaningful tokens after cleaning?*

- "I am not happy with this"

- Reason:
Despite preprocessing, it retains important semantic meaning, especially the word "not" which preserves sentiment.

---
## **Task 5: Frequency Analysis**

In [108]:
all_tokens = []

for sentence in test_sentences:
    tokens, _ = preprocess_text(sentence)
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10 Frequent Words:")
print(counter.most_common(10))

print("\nTop 5 Least Frequent Words:")
print(counter.most_common()[-5:])

Top 10 Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('looved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 Least Frequent Words:
[('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


---
## **Task 6: Build Full Pipeline**

In [109]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, cleaned = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(cleaned)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }


# Test pipeline
result = full_pipeline(test_sentences)

print(result)

print("\nClean Sentences:")
for i, sent in enumerate(result["clean_sentences"], 1):
    print(f"{i}. {sent}")

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'looved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'noo', 'this', 'baad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely looved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'noo this baad', 'got', 'win now limited offer', 'not happy with this']}

Clean Sentences:
1. get free access now
2. absolutely looved this product
3. worst service ever
4. call
5. this the best course
6. visit now
7. noo this baad
8. got
9. win now limited offer
10. not happy with this


---
## **Task 7: Error Handling**

In [110]:
edge_cases = ["", None, "😍😍🔥🔥", "1234567890", "!!!@@@", "ok", "No"]

for case in edge_cases:
    tokens, cleaned = preprocess_text(case)

    print("-" * 50)
    print(f"Input   : {case}")
    print(f"Tokens  : {tokens}")
    print(f"Cleaned : {cleaned}")

--------------------------------------------------
Input   : 
Tokens  : []
Cleaned : 
--------------------------------------------------
Input   : None
Tokens  : []
Cleaned : 
--------------------------------------------------
Input   : 😍😍🔥🔥
Tokens  : []
Cleaned : 
--------------------------------------------------
Input   : 1234567890
Tokens  : []
Cleaned : 
--------------------------------------------------
Input   : !!!@@@
Tokens  : []
Cleaned : 
--------------------------------------------------
Input   : ok
Tokens  : []
Cleaned : 
--------------------------------------------------
Input   : No
Tokens  : ['no']
Cleaned : no
